# Jenga Physics Simulator — Paper-Enhanced GPU Edition
## N=150 × 3 Friction Levels · Ziglar Mechanics · NVIDIA Spark · Full Floor Collapse

| Improvement | Description |
|---|---|
| **Ziglar Forces** | Fapp_min=3µmg (Eq.4), Fapp_torque=4µmg (Eq.8), real dimensions (8.1×2.6×1.8 cm) |
| **3 friction levels** | µ=0.25 (low), µ=0.40 (Ziglar nominal), µ=0.60 (high) |
| **Full floor collapse** | `simulate_until_rest()` — blocks reach the floor before the video ends |
| **GPU CuPy** | **Main process** uses GPU for CNN and AABB batch; workers are 100% CPU |
| **Correct multiprocessing** | `spawn` context + `jenga_worker.py` module — no `cudaErrorInitializationError` or `AttributeError: Can't get attribute 'run_episode'` |

### Why a separate `.py` module
With `spawn`, each worker re-imports `__main__`. In Jupyter, `__main__` is
`_frozen_importlib.BuiltinImporter` — not a file on disk — so functions
defined in the notebook **do not exist** from the worker's perspective.
The standard solution is to put all the code workers need into a real module
(`jenga_worker.py`) that workers can import.


In [ ]:
# ── Step 0: write jenga_worker.py to the working directory ────────────────────
# This must run BEFORE creating the executor.
# Spawn workers import this file; the notebook imports it too.
_WORKER_SRC = '"""\njenga_worker.py\n===============\nMódulo auxiliar para ProcessPoolExecutor con spawn context en Jupyter.\n\nPROBLEMA: con spawn, cada worker reimporta __main__. En Jupyter, __main__ es\n_frozen_importlib.BuiltinImporter (no un archivo .py), así que las funciones\ndefinidas en el notebook NO son accesibles desde los workers.\n\nSOLUCIÓN: todas las funciones que los workers necesitan viven aquí.\nEl notebook importa desde este archivo; los workers también pueden hacerlo.\n"""\n\nimport os, json, warnings\nimport numpy as np\nfrom scipy.spatial import ConvexHull, Delaunay\n\nwarnings.filterwarnings("ignore", category=UserWarning)\n\n# ── Physics constants (workers cannot read the Jupyter kernel) ─────────────\nGPU_AVAILABLE   = False   # workers nunca usan CUDA\nNUMBA_AVAILABLE = False\ntry:\n    from numba import njit\n    NUMBA_AVAILABLE = True\nexcept ImportError:\n    pass\n\nG_ACCEL             = 9.81\nBLOCK_MASS_KG       = 0.0196\nFAPP_MIN_FACTOR     = 3.0\nFAPP_TORQUE_FACTOR  = 4.0\nL_BLOCK = 0.081\nW_BLOCK = 0.026\nH_BLOCK = 0.018\nGAP     = 0.0003\nQUAT_Y  = np.array([np.cos(np.pi/4), 0., 0., np.sin(np.pi/4)])\nQUAT_X  = np.array([1., 0., 0., 0.])\nL       = L_BLOCK\n\n# ── Simulation parameters (overridden by args when passed) ─────────────────\nSOLVER_ITERS        = 14\nN_SUBSTEPS          = 3\nFULL_FALL_STEPS     = 400\nFULL_FALL_KE_THRESH = 1e-7\nFULL_FALL_FRAME_SKIP= 4\nDISP_THRESH_FRAC    = 0.5\nKE_THRESH           = 5e-5\nANGLE_THRESH_DEG    = 30\nEPS                 = 1e-9\n\n# ═════════════════════════════════════════════════════════════════════════════\n# MATHUTILS\n# ═════════════════════════════════════════════════════════════════════════════\ndef fast_cross(a, b):\n    return np.array((a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0]))\n\ndef quat_normalize(q):\n    n = np.linalg.norm(q)\n    return q/n if n > 1e-12 else np.array([1.,0.,0.,0.])\n\ndef quat_to_rotmat(q):\n    w,x,y,z = q\n    return np.array([[1-2*(y*y+z*z), 2*(x*y-z*w),   2*(x*z+y*w)],\n                     [2*(x*y+z*w),   1-2*(x*x+z*z), 2*(y*z-x*w)],\n                     [2*(x*z-y*w),   2*(y*z+x*w),   1-2*(x*x+y*y)]])\n\ndef quat_multiply(q1, q2):\n    w1,x1,y1,z1=q1; w2,x2,y2,z2=q2\n    return np.array([w1*w2-x1*x2-y1*y2-z1*z2, w1*x2+x1*w2+y1*z2-z1*y2,\n                     w1*y2-x1*z2+y1*w2+z1*x2, w1*z2+x1*y2-y1*x2+z1*w2])\n\ndef quat_integrate(q, omega, dt):\n    oq = np.array([0., omega[0], omega[1], omega[2]])\n    return quat_normalize(q + 0.5*quat_multiply(oq,q)*dt)\n\ndef box_inertia_body(mass, lx, ly, lz):\n    return np.diag([(mass/12.)*(ly**2+lz**2),\n                    (mass/12.)*(lx**2+lz**2),\n                    (mass/12.)*(lx**2+ly**2)])\n\ndef box_corners(center, half_extents, R):\n    hx,hy,hz = half_extents\n    local = np.array([[hx,hy,hz],[hx,hy,-hz],[hx,-hy,hz],[hx,-hy,-hz],\n                      [-hx,hy,hz],[-hx,hy,-hz],[-hx,-hy,hz],[-hx,-hy,-hz]])\n    return center + local @ R.T\n\ndef cpu_aabb_pairs(positions, half_extents, margin=0.002):\n    """AABB broadphase 100% CPU — only method used in workers."""\n    N = len(positions)\n    pairs = []\n    for i in range(N):\n        for j in range(i+1, N):\n            sep = np.abs(positions[i]-positions[j])\n            ext = half_extents[i]+half_extents[j]+margin\n            if np.all(sep <= ext):\n                pairs.append((i, j))\n    return pairs\n\n# ═════════════════════════════════════════════════════════════════════════════\n# RIGID BODY\n# ═════════════════════════════════════════════════════════════════════════════\nclass RigidBody:\n    def __init__(self, body_id, position, half_extents, mass,\n                 quat=None, static=False, meta=None, mu_static=0.40):\n        self.id          = body_id\n        self.position    = np.array(position, dtype=float)\n        self.half_extents= np.array(half_extents, dtype=float)\n        self.quat        = np.array([1.,0.,0.,0.]) if quat is None else np.array(quat, float)\n        self.velocity    = np.zeros(3)\n        self.omega       = np.zeros(3)\n        self.static      = static\n        self.mass        = mass\n        self.inv_mass    = 0. if static else 1./mass\n        lx,ly,lz        = 2*self.half_extents\n        self.inertia_body     = box_inertia_body(mass, lx, ly, lz)\n        self.inv_inertia_body = np.zeros((3,3)) if static else np.linalg.inv(self.inertia_body)\n        self.friction    = mu_static\n        self.restitution = 0.10\n        self.active      = True\n        self.asleep      = False\n        self.meta        = meta or {}\n        self.init_position = self.position.copy()\n        self.init_quat     = self.quat.copy()\n        self.on_floor      = False\n        slot             = self.meta.get("slot", 1)\n        self.block_type  = "center" if slot == 1 else "side"\n        self.fapp_min    = FAPP_MIN_FACTOR    * mu_static * mass * G_ACCEL\n        self.fapp_torque = FAPP_TORQUE_FACTOR * mu_static * mass * G_ACCEL\n\n    def world_inertia_inv(self):\n        if self.static: return np.zeros((3,3))\n        R = quat_to_rotmat(self.quat)\n        return R @ self.inv_inertia_body @ R.T\n\n    def corners(self):\n        return box_corners(self.position, self.half_extents, quat_to_rotmat(self.quat))\n\n    def kinetic_energy(self):\n        if self.static or not self.active: return 0.\n        Iw_inv = self.world_inertia_inv()\n        Iw = np.linalg.pinv(Iw_inv) if np.linalg.matrix_rank(Iw_inv)>0 else np.zeros((3,3))\n        return 0.5*self.mass*(self.velocity@self.velocity) + 0.5*(self.omega@Iw@self.omega)\n\n    def is_resting_on_floor(self, floor_z=0., eps=0.003):\n        bottom_z = self.position[2]-self.half_extents[2]\n        speed = np.linalg.norm(self.velocity)+np.linalg.norm(self.omega)\n        return bottom_z <= floor_z+eps and speed < 0.02\n\n# ═════════════════════════════════════════════════════════════════════════════\n# COLLISION\n# ═════════════════════════════════════════════════════════════════════════════\nclass Contact:\n    __slots__ = ("a","b","normal","point","penetration")\n    def __init__(self,a,b,normal,point,penetration):\n        self.a=a;self.b=b;self.normal=normal;self.point=point;self.penetration=penetration\n\ndef floor_contacts(body, floor_z=0.):\n    return [Contact(None,body,np.array([0.,0.,1.]),c.copy(),floor_z-c[2])\n            for c in body.corners() if c[2] < floor_z]\n\ndef _obb_axes(R): return [R[:,0],R[:,1],R[:,2]]\n\ndef _project_obb(center, axes, hext, axis):\n    r = sum(hext[i]*abs(np.dot(axes[i],axis)) for i in range(3))\n    return np.dot(center,axis), r\n\ndef sat_box_box(a, b):\n    Ra=quat_to_rotmat(a.quat); Rb=quat_to_rotmat(b.quat)\n    aa=_obb_axes(Ra); ab=_obb_axes(Rb)\n    cands = list(aa)+list(ab)\n    for ea in aa:\n        for eb in ab:\n            ax=fast_cross(ea,eb); n=np.linalg.norm(ax)\n            if n>1e-6: cands.append(ax/n)\n    min_pen=np.inf; min_axis=None\n    for axis in cands:\n        ca,ra=_project_obb(a.position,aa,a.half_extents,axis)\n        cb,rb=_project_obb(b.position,ab,b.half_extents,axis)\n        ov=ra+rb-abs(ca-cb)\n        if ov<=0: return None\n        if ov<min_pen:\n            min_pen=ov; min_axis=axis*(1. if cb>=ca else -1.)\n    if min_axis is None: return None\n    return min_axis/(np.linalg.norm(min_axis)+EPS), min_pen\n\ndef aabb_overlap(a, b, margin=0.001):\n    return np.all(np.abs(a.position-b.position) <= a.half_extents+b.half_extents+margin)\n\ndef _ch2d(pts):\n    if len(pts)<3: return pts\n    try: h=ConvexHull(pts); return pts[h.vertices]\n    except: return pts\n\ndef _sh(subject, clip):\n    def ins(p,a,b): return (b[0]-a[0])*(p[1]-a[1])-(b[1]-a[1])*(p[0]-a[0])>=-1e-12\n    def ix(p1,p2,a,b):\n        x1,y1=p1;x2,y2=p2;x3,y3=a;x4,y4=b\n        d=(x1-x2)*(y3-y4)-(y1-y2)*(x3-x4)\n        if abs(d)<1e-12: return p2\n        t=((x1-x3)*(y3-y4)-(y1-y3)*(x3-x4))/d\n        return np.array([x1+t*(x2-x1),y1+t*(y2-y1)])\n    out=list(subject)\n    for i in range(len(clip)):\n        a,b=clip[i],clip[(i+1)%len(clip)]; inp,out=out,[]\n        if not inp: break\n        for j in range(len(inp)):\n            cur,prev=inp[j],inp[j-1]\n            if ins(cur,a,b):\n                if not ins(prev,a,b): out.append(ix(prev,cur,a,b))\n                out.append(cur)\n            elif ins(prev,a,b): out.append(ix(prev,cur,a,b))\n    return np.array(out) if out else np.zeros((0,2))\n\ndef _face_clip(a, b, normal, mp=4):\n    idx=int(np.argmax(np.abs(normal))); oth=[i for i in range(3) if i!=idx]\n    sign=1. if normal[idx]>=0 else -1.\n    ca,cb=a.corners(),b.corners()\n    inter=_sh(_ch2d(ca[:,oth]),_ch2d(cb[:,oth]))\n    if len(inter)<3: return None\n    if len(inter)>mp:\n        ctr=inter.mean(0); ang=np.arctan2(inter[:,1]-ctr[1],inter[:,0]-ctr[0])\n        sel=np.linspace(0,len(inter),mp,endpoint=False).astype(int)\n        inter=inter[np.argsort(ang)][sel]\n    ea=np.zeros(3); ea[idx]=1.\n    w=0.5*((ca@ea).max()+(cb@ea).min()) if sign>0 else 0.5*((ca@ea).min()+(cb@ea).max())\n    pts=np.zeros((len(inter),3)); pts[:,oth]=inter; pts[:,idx]=w\n    return list(pts)\n\ndef manifold_pts(a, b, normal):\n    idx=int(np.argmax(np.abs(normal)))\n    if abs(normal[idx])>0.9:\n        fp=_face_clip(a,b,normal)\n        if fp is not None: return fp\n    ca=a.corners(); return list(ca[np.argsort(ca@normal)[-2:]])\n\ndef box_box_contacts(a, b):\n    res=sat_box_box(a,b)\n    if res is None: return []\n    normal,pen=res; pts=manifold_pts(a,b,normal)\n    return [Contact(a,b,normal,p,pen/max(len(pts),1)) for p in pts]\n\n# ═════════════════════════════════════════════════════════════════════════════\n# SOLVER (PGS, Numba JIT si disponible)\n# ═════════════════════════════════════════════════════════════════════════════\nif NUMBA_AVAILABLE:\n    from numba import njit as _njit\n    import numpy as _npn\n\n    @_njit(cache=True)\n    def _cross3(a,b):\n        return _npn.array([a[1]*b[2]-a[2]*b[1],a[2]*b[0]-a[0]*b[2],a[0]*b[1]-a[1]*b[0]])\n\n    @_njit(cache=True)\n    def _mv3(M,v):\n        return _npn.array([M[0,0]*v[0]+M[0,1]*v[1]+M[0,2]*v[2],\n                           M[1,0]*v[0]+M[1,1]*v[1]+M[1,2]*v[2],\n                           M[2,0]*v[0]+M[2,1]*v[1]+M[2,2]*v[2]])\n\n    @_njit(cache=True)\n    def _pgs(iters,ai,bi,ns,t1s,t2s,kns,kt1,kt2,es,mus,ras,rbs,\n             jn,jt1,jt2,vel,ome,im,ii):\n        NC=len(ai)\n        for _ in range(iters):\n            for c in range(NC):\n                a_=ai[c]; b_=bi[c]; n=ns[c]; ra=ras[c]; rb=rbs[c]\n                va=vel[a_]+_cross3(ome[a_],ra) if a_>=0 else _npn.zeros(3)\n                vb=vel[b_]+_cross3(ome[b_],rb)\n                vn=(vb-va)@n; djn=-(1+es[c])*vn/kns[c]\n                na_=max(jn[c]+djn,0.); dj=na_-jn[c]; jn[c]=na_\n                imp=dj*n\n                if a_>=0: vel[a_]-=imp*im[a_]; ome[a_]-=_mv3(ii[a_],_cross3(ra,imp))\n                vel[b_]+=imp*im[b_]; ome[b_]+=_mv3(ii[b_],_cross3(rb,imp))\n                for t,kt,jt_ in[(t1s[c],kt1[c],jt1),(t2s[c],kt2[c],jt2)]:\n                    va2=vel[a_]+_cross3(ome[a_],ra) if a_>=0 else _npn.zeros(3)\n                    vb2=vel[b_]+_cross3(ome[b_],rb)\n                    vt=(vb2-va2)@t; djt=-vt/kt; mx=mus[c]*jn[c]\n                    p=jt_[c]; nj=max(-mx,min(p+djt,mx)); dj2=nj-p; jt_[c]=nj\n                    im2=dj2*t\n                    if a_>=0: vel[a_]-=im2*im[a_]; ome[a_]-=_mv3(ii[a_],_cross3(ra,im2))\n                    vel[b_]+=im2*im[b_]; ome[b_]+=_mv3(ii[b_],_cross3(rb,im2))\n\n\ndef _tangents(n):\n    t1 = fast_cross(n, np.array([1.,0.,0.]) if abs(n[0])<0.9 else np.array([0.,1.,0.]))\n    t1 /= np.linalg.norm(t1)+1e-12\n    t2 = fast_cross(n,t1); t2 /= np.linalg.norm(t2)+1e-12\n    return t1, t2\n\n\ndef solve_contacts(contacts, dt, iterations=None, warm_cache=None):\n    if iterations is None: iterations = SOLVER_ITERS\n    if not contacts: return\n    bl = []; seen = set()\n    for c in contacts:\n        if c.a and id(c.a) not in seen: bl.append(c.a); seen.add(id(c.a))\n        if id(c.b) not in seen: bl.append(c.b); seen.add(id(c.b))\n    im_ = {id(b):i for i,b in enumerate(bl)}\n    vel=np.array([b.velocity for b in bl]); ome=np.array([b.omega for b in bl])\n    inv_m=np.array([b.inv_mass for b in bl]); inv_i=np.array([b.world_inertia_inv() for b in bl])\n    NC=len(contacts)\n    ai=np.zeros(NC,int); bi=np.zeros(NC,int)\n    ns=np.zeros((NC,3)); t1s=np.zeros((NC,3)); t2s=np.zeros((NC,3))\n    kns=np.zeros(NC); kt1=np.zeros(NC); kt2=np.zeros(NC); es=np.zeros(NC); mus=np.zeros(NC)\n    ras=np.zeros((NC,3)); rbs=np.zeros((NC,3))\n    jn=np.zeros(NC); jt1=np.zeros(NC); jt2=np.zeros(NC)\n    keys=[]\n    for ci,c in enumerate(contacts):\n        a_=im_.get(id(c.a),-1) if c.a else -1; b_=im_[id(c.b)]\n        n=c.normal; t1,t2=_tangents(n)\n        ra=c.point-(c.a.position if c.a else np.zeros(3)); rb=c.point-c.b.position\n        Ia=inv_i[a_] if a_>=0 else np.zeros((3,3)); Ib=inv_i[b_]\n        ima=inv_m[a_] if a_>=0 else 0.\n        def _k(t_,Ia_=Ia,Ib_=Ib,ra_=ra,rb_=rb,ima_=ima,b__=b_):\n            ca=fast_cross(Ia_@fast_cross(ra_,t_),ra_) if a_>=0 else np.zeros(3)\n            cb=fast_cross(Ib_@fast_cross(rb_,t_),rb_)\n            return ima_+inv_m[b__]+np.dot(ca+cb,t_)+1e-12\n        mu=0.5*(c.a.friction if c.a else 0.4)+0.5*c.b.friction\n        e=0.5*((c.a.restitution if c.a else 0.1)+c.b.restitution)\n        key=(id(c.a),id(c.b)) if c.a else (None,id(c.b))\n        jn0,jt1_0,jt2_0=warm_cache.get(key,(0.,0.,0.)) if warm_cache else (0.,0.,0.)\n        ai[ci]=a_;bi[ci]=b_;ns[ci]=n;t1s[ci]=t1;t2s[ci]=t2\n        kns[ci]=_k(n);kt1[ci]=_k(t1);kt2[ci]=_k(t2);es[ci]=e;mus[ci]=mu\n        ras[ci]=ra;rbs[ci]=rb;jn[ci]=jn0;jt1[ci]=jt1_0;jt2[ci]=jt2_0\n        keys.append(key)\n    if NUMBA_AVAILABLE:\n        _pgs(iterations,ai,bi,ns,t1s,t2s,kns,kt1,kt2,es,mus,ras,rbs,jn,jt1,jt2,vel,ome,inv_m,inv_i)\n    else:\n        for _ in range(iterations):\n            for ci in range(NC):\n                a_=int(ai[ci]);b_=int(bi[ci]);n=ns[ci];ra=ras[ci];rb=rbs[ci]\n                va=vel[a_]+np.cross(ome[a_],ra) if a_>=0 else np.zeros(3)\n                vb=vel[b_]+np.cross(ome[b_],rb)\n                vn=np.dot(vb-va,n); djn=-(1+es[ci])*vn/kns[ci]\n                na_=max(jn[ci]+djn,0.); dj_=na_-jn[ci]; jn[ci]=na_\n                imp=dj_*n\n                if a_>=0: vel[a_]-=imp*inv_m[a_]; ome[a_]-=inv_i[a_]@np.cross(ra,imp)\n                vel[b_]+=imp*inv_m[b_]; ome[b_]+=inv_i[b_]@np.cross(rb,imp)\n                for t,kt,jt_ in[(t1s[ci],kt1[ci],jt1),(t2s[ci],kt2[ci],jt2)]:\n                    va2=vel[a_]+np.cross(ome[a_],ra) if a_>=0 else np.zeros(3)\n                    vb2=vel[b_]+np.cross(ome[b_],rb)\n                    vt=np.dot(vb2-va2,t); djt=-vt/kt; mx=mus[ci]*jn[ci]\n                    p=jt_[ci]; nj=np.clip(p+djt,-mx,mx); dj2=nj-p; jt_[ci]=nj\n                    im2=dj2*t\n                    if a_>=0: vel[a_]-=im2*inv_m[a_]; ome[a_]-=inv_i[a_]@np.cross(ra,im2)\n                    vel[b_]+=im2*inv_m[b_]; ome[b_]+=inv_i[b_]@np.cross(rb,im2)\n    ss={id(b) for b in bl if b.static or not b.active}\n    for i,b in enumerate(bl):\n        if id(b) not in ss: b.velocity=vel[i].copy(); b.omega=ome[i].copy()\n    if warm_cache is not None:\n        warm_cache.clear()\n        for ci,k in enumerate(keys):\n            if k: warm_cache[k]=(float(jn[ci]),float(jt1[ci]),float(jt2[ci]))\n    # position correction\n    seen2={}\n    for c in contacts:\n        k=(id(c.a) if c.a else -1,id(c.b))\n        if k not in seen2 or c.penetration>seen2[k].penetration: seen2[k]=c\n    for c in seen2.values():\n        a,b=c.a,c.b; mag=max(c.penetration-0.0004,0.)*0.35\n        if mag<=0: continue\n        ima_=a.inv_mass if a else 0.; tot=ima_+b.inv_mass\n        if tot<=0: continue\n        corr=c.normal*(mag/tot)\n        if a and not a.static and a.active: a.position-=corr*ima_\n        if not b.static and b.active: b.position+=corr*b.inv_mass\n\n# ═════════════════════════════════════════════════════════════════════════════\n# WORLD\n# ═════════════════════════════════════════════════════════════════════════════\nclass World:\n    GRAVITY = np.array([0.,0.,-9.81])\n\n    def __init__(self, floor_z=0., dt=1./240., solver_iterations=None, n_substeps=None):\n        self.bodies=[]; self.floor_z=floor_z\n        self.dt=dt\n        self.solver_iterations=solver_iterations or SOLVER_ITERS\n        self.n_substeps=n_substeps or N_SUBSTEPS\n        self.time=0.; self.step_count=0; self._wc={}\n\n    def add_body(self,b): self.bodies.append(b)\n    def active_dynamic(self): return [b for b in self.bodies if not b.static and b.active]\n    def total_ke(self): return sum(b.kinetic_energy() for b in self.bodies)\n    def wake_all(self):\n        for b in self.bodies: b.asleep=False\n\n    def step(self):\n        sub=self.dt/self.n_substeps\n        for _ in range(self.n_substeps): self._sub(sub)\n        self.time+=self.dt; self.step_count+=1\n\n    def _sub(self, dt):\n        dyn=self.active_dynamic()\n        for b in dyn:\n            if not getattr(b,"asleep",False): b.velocity+=self.GRAVITY*dt\n        contacts=[]\n        for b in dyn: contacts.extend(floor_contacts(b,self.floor_z))\n        ad=[b for b in self.bodies if not b.static and b.active]\n        if len(ad)>6:\n            pos=np.array([b.position for b in ad])\n            hext=np.array([b.half_extents for b in ad])\n            for i,j in cpu_aabb_pairs(pos,hext):\n                bi_,bj_=ad[i],ad[j]\n                if getattr(bi_,"asleep",False) and getattr(bj_,"asleep",False): continue\n                contacts.extend(box_box_contacts(bi_,bj_))\n        else:\n            for i,bi_ in enumerate(self.bodies):\n                if bi_.static or not bi_.active: continue\n                for bj_ in self.bodies[i+1:]:\n                    if bj_.static or not bj_.active: continue\n                    if getattr(bi_,"asleep",False) and getattr(bj_,"asleep",False): continue\n                    if not aabb_overlap(bi_,bj_): continue\n                    contacts.extend(box_box_contacts(bi_,bj_))\n        solve_contacts(contacts,dt,self.solver_iterations,self._wc)\n        for b in dyn:\n            if getattr(b,"asleep",False): continue\n            b.position=b.position+b.velocity*dt\n            b.quat=quat_integrate(b.quat,b.omega,dt)\n            b.velocity*=0.9995; b.omega*=0.999\n            if np.linalg.norm(b.velocity)+np.linalg.norm(b.omega)<0.008: b.asleep=True\n\n    def simulate_until_rest(self, max_steps=None, ke_thresh=None, frame_skip=None):\n        if max_steps is None:  max_steps  = FULL_FALL_STEPS\n        if ke_thresh is None:  ke_thresh  = FULL_FALL_KE_THRESH\n        if frame_skip is None: frame_skip = FULL_FALL_FRAME_SKIP\n        frames=[]\n        for si in range(max_steps):\n            self.step()\n            if si%frame_skip==0: frames.append(_snapshot(self.active_dynamic()))\n            if self.total_ke()<ke_thresh: break\n        for b in self.active_dynamic(): b.on_floor=b.is_resting_on_floor(self.floor_z)\n        return frames\n\n# ═════════════════════════════════════════════════════════════════════════════\n# TOWER\n# ═════════════════════════════════════════════════════════════════════════════\ndef build_tower(world=None, dt=1./240., jitter=0., rng=None,\n                n_layers=18, mu_static=0.40):\n    if world is None: world=World(dt=dt)\n    if rng is None:   rng=np.random.default_rng()\n    world.add_body(RigidBody("floor",[0,0,-0.5],[5,5,0.5],mass=1.,static=True,meta={"role":"floor"}))\n    blocks=[]\n    for layer in range(n_layers):\n        z=H_BLOCK*layer+H_BLOCK/2; ax=(layer%2==0)\n        quat=QUAT_X if ax else QUAT_Y\n        for slot in range(3):\n            off=(slot-1)*(W_BLOCK+GAP)\n            pos=np.array([0.,off,z]) if ax else np.array([off,0.,z])\n            if jitter>0: pos=pos+rng.normal(0,jitter,3)*np.array([1.,1.,0.2])\n            bid=f"L{layer:02d}_{slot}"\n            b=RigidBody(bid,pos,[L_BLOCK/2,W_BLOCK/2,H_BLOCK/2],\n                        mass=BLOCK_MASS_KG,quat=quat,\n                        meta={"layer":layer,"slot":slot,"id":bid},\n                        mu_static=mu_static)\n            world.add_body(b); blocks.append(b)\n    return world, blocks\n\ndef removable_blocks(blocks, top_protected=2):\n    ml=max(b.meta["layer"] for b in blocks if b.active)\n    return [b for b in blocks if b.active and b.meta["layer"]<=ml-top_protected]\n\ndef tower_center_of_mass(blocks):\n    ac=[b for b in blocks if b.active]\n    if not ac: return np.zeros(3)\n    tm=sum(b.mass for b in ac)\n    return sum(b.mass*b.position for b in ac)/tm\n\n# ═════════════════════════════════════════════════════════════════════════════\n# STABILITY + ZIGLAR\n# ═════════════════════════════════════════════════════════════════════════════\ndef support_polygon_xy(blocks, floor_z=0., eps=0.004):\n    pts=[]\n    for b in blocks:\n        if not b.active: continue\n        if b.position[2]-b.half_extents[2]<=floor_z+eps: pts.extend(b.corners()[:,:2])\n    if len(pts)<3: return None\n    pts=np.array(pts)\n    try: h=ConvexHull(pts); return pts[h.vertices]\n    except: return None\n\ndef is_statically_stable(blocks, floor_z=0.):\n    ac=[b for b in blocks if b.active]\n    if not ac: return True,np.zeros(2),None,0.\n    tm=sum(b.mass for b in ac); com=sum(b.mass*b.position for b in ac)/tm\n    cxy=com[:2]; poly=support_polygon_xy(blocks,floor_z)\n    if poly is None or len(poly)<3: return False,cxy,poly,-1.\n    inside=Delaunay(poly).find_simplex(cxy)>=0\n    n=len(poly); md=np.inf\n    for i in range(n):\n        a=poly[i];b=poly[(i+1)%n]; e=b-a; el=np.linalg.norm(e)\n        if el<1e-9: continue\n        ed=e/el; nor=np.array([ed[1],-ed[0]])\n        md=min(md,-np.dot(cxy-a,nor))\n    return bool(inside),cxy,poly,float(md)\n\ndef ziglar_block_analysis(target, blocks, move_axis="y"):\n    mu=target.friction; m=target.mass\n    slot=target.meta["slot"]; layer=target.meta["layer"]\n    fmin=FAPP_MIN_FACTOR*mu*m*G_ACCEL; ftor=FAPP_TORQUE_FACTOR*mu*m*G_ACCEL\n    same=[b for b in blocks if b.active and b.meta["layer"]==layer and b.id!=target.id]\n    nr=len(same)\n    if slot==1:\n        risk="low" if nr>=2 else "medium"; mt="center_xaxis"; ct=False\n    elif move_axis=="y":\n        risk="low" if nr>=1 else "high"; mt="side_yaxis"; ct=False\n    else:\n        other=[b for b in same if b.meta["slot"]!=1]\n        risk="high" if len(other)==0 else "medium"; mt="side_xaxis"; ct=True\n    return dict(fapp_min_mN=fmin*1000,fapp_torque_mN=ftor*1000,\n                risk=risk,move_type=mt,can_torque=ct,n_remaining=nr,mu=mu)\n\ndef _quat_angle_deg(q1,q2):\n    return np.degrees(2*np.arccos(min(abs(np.dot(q1,q2)),1.)))\n\ndef composite_collapse_check(blocks, rest_pos, rest_q,\n                              disp_thresh=None, ke_thresh=None, angle_thresh=None):\n    if disp_thresh  is None: disp_thresh  = L_BLOCK*DISP_THRESH_FRAC\n    if ke_thresh    is None: ke_thresh    = KE_THRESH\n    if angle_thresh is None: angle_thresh = ANGLE_THRESH_DEG\n    ac=[b for b in blocks if b.active]\n    ke=sum(b.kinetic_energy() for b in ac)\n    md=0.; ma=0.\n    for b in ac:\n        pp=rest_pos.get(b.id)\n        if pp is not None: md=max(md,np.linalg.norm(b.position-pp))\n        pq=rest_q.get(b.id)\n        if pq is not None: ma=max(ma,_quat_angle_deg(b.quat,pq))\n    cd=md>disp_thresh; ck=ke>ke_thresh; ca=ma>angle_thresh\n    return cd and ck and ca, dict(kinetic_energy=float(ke),max_displacement=float(md),\n                                   max_angle_deg=float(ma),cond_disp=bool(cd),\n                                   cond_ke=bool(ck),cond_angle=bool(ca))\n\n# ═════════════════════════════════════════════════════════════════════════════\n# SNAPSHOT (necesario en workers)\n# ═════════════════════════════════════════════════════════════════════════════\ndef _snapshot(blocks_list):\n    return [dict(id=b.id,position=b.position.tolist(),quat=b.quat.tolist(),\n                  half_extents=b.half_extents.tolist(),slot=b.meta.get("slot",0))\n            for b in blocks_list if b.active]\n\n# ═════════════════════════════════════════════════════════════════════════════\n# run_episode — función top-level, pickleable para spawn workers\n# ═════════════════════════════════════════════════════════════════════════════\ndef run_episode(args):\n    """\n    Episodio completo. Debe ser función top-level en un módulo real\n    para que spawn workers puedan importarla y hacer pickle de ella.\n    """\n    (exp_id, n_layers, settle_steps, solver_iters, n_substeps,\n     seed, data_dir, mu_level_name, mu_value) = args\n\n    rng=np.random.default_rng(seed)\n    world=World(dt=1./240.,solver_iterations=solver_iters,n_substeps=n_substeps)\n    world,blocks=build_tower(world,n_layers=n_layers,mu_static=mu_value)\n\n    rest_pos={b.id:b.position.copy() for b in blocks}\n    rest_q  ={b.id:b.quat.copy()    for b in blocks}\n    rounds=[]; frame_snaps=[]; removed=[]; collapsed=False; fall_frames=[]\n\n    for ri in range(999):\n        rem=removable_blocks(blocks)\n        if not rem: break\n        target=rem[rng.integers(0,len(rem))]\n        move_axis="y" if rng.random()>0.5 else "x"\n        zig=ziglar_block_analysis(target,blocks,move_axis)\n        target.active=False; removed.append(target.id); world.wake_all()\n        rest_pos={b.id:b.position.copy() for b in blocks if b.active}\n        rest_q  ={b.id:b.quat.copy()    for b in blocks if b.active}\n        stable,cxy,poly,margin=is_statically_stable(blocks)\n        rf=[]\n        for s in range(settle_steps):\n            world.step()\n            if s%5==0: rf.append(_snapshot([b for b in blocks if b.active]))\n        frame_snaps.append(rf)\n        col,met=composite_collapse_check(blocks,rest_pos,rest_q)\n        com=tower_center_of_mass(blocks)\n        rounds.append(dict(\n            round=ri,removed_block=target.id,\n            removed_layer=int(target.meta["layer"]),\n            removed_slot=int(target.meta["slot"]),\n            block_type=target.block_type,move_axis=move_axis,\n            analytic_stable=bool(stable),analytic_margin=float(margin),\n            collapsed=bool(col),com=com.tolist(),\n            n_active=int(sum(1 for b in blocks if b.active)),\n            positions={b.id:b.position.tolist() for b in blocks if b.active},\n            quats    ={b.id:b.quat.tolist()     for b in blocks if b.active},\n            ziglar_risk=zig["risk"],ziglar_move_type=zig["move_type"],\n            ziglar_can_torque=zig["can_torque"],\n            ziglar_fapp_min_mN=zig["fapp_min_mN"],\n            ziglar_fapp_torque_mN=zig["fapp_torque_mN"],\n            **met))\n        if col:\n            fall_frames=world.simulate_until_rest()\n            collapsed=True; break\n\n    result=dict(\n        exp_id=exp_id,mu_level=mu_level_name,mu_value=float(mu_value),\n        n_layers=n_layers,n_blocks_initial=len(blocks),\n        collapsed=bool(collapsed),n_rounds=len(rounds),\n        removed_order=removed,rounds=rounds,\n        frame_snapshots=frame_snaps,fall_frames=fall_frames,\n        n_blocks_on_floor=int(sum(1 for b in blocks if b.active and b.on_floor)),\n        wall_time_s=0.)\n    os.makedirs(f"{data_dir}/experiments",exist_ok=True)\n    slim={k:v for k,v in result.items() if k not in ("frame_snapshots","fall_frames")}\n    with open(f"{data_dir}/experiments/exp_{mu_level_name}_{exp_id:04d}.json","w") as f:\n        json.dump(slim,f)\n    return result\n\ndef _run_and_track(args):\n    try: return run_episode(args), args[0]\n    except Exception as e:\n        import traceback; traceback.print_exc()\n        print(f"  ⚠ exp {args[0]} [{args[7]}] failed: {e}")\n        return None, args[0]\n'

import os, sys
_worker_path = os.path.join(os.getcwd(), "jenga_worker.py")
with open(_worker_path, "w") as _f:
    _f.write(_WORKER_SRC)
print(f"jenga_worker.py written to: {_worker_path}")

# Make sure the current directory is in sys.path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Force re-import if already cached
if "jenga_worker" in sys.modules:
    del sys.modules["jenga_worker"]

import jenga_worker as _jw
print(f"jenga_worker importado OK  (Numba: {_jw.NUMBA_AVAILABLE})")


In [ ]:
import os, sys, json, time as _time_mod, glob, pickle, warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull, Delaunay
warnings.filterwarnings("ignore", category=UserWarning)

# ── GPU — MAIN PROCESS ONLY ───────────────────────────────────────────────────
GPU_AVAILABLE = False; gpu_name = "none"
try:
    import cupy as cp
    _ = cp.array([1.,2.,3.])
    GPU_AVAILABLE = True
    gpu_name = cp.cuda.runtime.getDeviceProperties(0)["name"].decode()
    print(f"✓ GPU: {gpu_name}  CuPy {cp.__version__}")
except Exception as _e:
    print(f"GPU/CuPy not available ({_e.__class__.__name__}) — CPU mode")
    print("  → pip install cupy-cuda12x")

NUMBA_AVAILABLE = _jw.NUMBA_AVAILABLE

_N_CPUS = os.cpu_count() or 1
try:
    import psutil; _RAM_GB = psutil.virtual_memory().total//(1024**3)
except: _RAM_GB = 8
print(f"CPUs: {_N_CPUS}  RAM: {_RAM_GB} GB  GPU: {GPU_AVAILABLE}  Numba: {NUMBA_AVAILABLE}")

# ══════════════════════════════════════════════════════════════════════════════
DEMO_MODE    = True
DEMO_NLAYERS = 6
FULL_NLAYERS = 18
N_EXPERIMENTS = 150

FRICTION_LEVELS = {"low": 0.25, "nominal": 0.40, "high": 0.60}

BLOCK_MASS_KG      = 0.0196
FAPP_MIN_FACTOR    = 3.0
FAPP_TORQUE_FACTOR = 4.0
G_ACCEL            = 9.81

if _N_CPUS == 1:
    N_WORKERS = 1; DEMO_SETTLE=50; FULL_SETTLE=180; N_SUBSTEPS=2; SOLVER_ITERS=12
else:
    N_WORKERS = min(_N_CPUS-1, 8); DEMO_SETTLE=70; FULL_SETTLE=220; N_SUBSTEPS=3; SOLVER_ITERS=16

FULL_FALL_STEPS=400; FULL_FALL_KE_THRESH=1e-7; FULL_FALL_FRAME_SKIP=4
DISP_THRESH_FRAC=0.5; KE_THRESH=5e-5; ANGLE_THRESH_DEG=30
IMG_SIZE=48 if _N_CPUS==1 else 64; BATCH_SIZE=16; EPOCHS=30; LR=3e-4; SEED=42
N_LAYERS = DEMO_NLAYERS if DEMO_MODE else FULL_NLAYERS
SETTLE   = DEMO_SETTLE  if DEMO_MODE else FULL_SETTLE

for d in ["data/experiments","data/frames","data/videos","data/dashboard","models"]:
    os.makedirs(d, exist_ok=True)

print(f"\nDEMO_MODE={DEMO_MODE}  N_LAYERS={N_LAYERS} ({N_LAYERS*3} blocks)")
print(f"Episodes: {N_EXPERIMENTS} × {len(FRICTION_LEVELS)} = {N_EXPERIMENTS*len(FRICTION_LEVELS)} total")
print(f"N_WORKERS={N_WORKERS}  SETTLE={SETTLE}  N_SUBSTEPS={N_SUBSTEPS}")
print(f"\nZiglar thresholds:")
for n,mu in FRICTION_LEVELS.items():
    fm=FAPP_MIN_FACTOR*mu*BLOCK_MASS_KG*G_ACCEL*1000
    ft=FAPP_TORQUE_FACTOR*mu*BLOCK_MASS_KG*G_ACCEL*1000
    print(f"  µ={mu:.2f} ({n:7s}): Fapp_min={fm:.1f} mN  Fapp_torque={ft:.1f} mN")


## 1. Import Physics from jenga_worker.py

In [ ]:
# Import all functions/classes into the notebook namespace
# so that subsequent cells can use them directly
from jenga_worker import (
    # math
    fast_cross, quat_normalize, quat_to_rotmat, quat_multiply,
    quat_integrate, box_inertia_body, box_corners,
    # physics
    RigidBody, Contact, World,
    # collision
    floor_contacts, sat_box_box, aabb_overlap, box_box_contacts,
    # tower
    build_tower, removable_blocks, tower_center_of_mass,
    # stability
    is_statically_stable, ziglar_block_analysis,
    composite_collapse_check,
    # snapshot / episode
    _snapshot, run_episode, _run_and_track,
)

# Sync worker constants with notebook values
import jenga_worker as _jw
_jw.SOLVER_ITERS        = SOLVER_ITERS
_jw.N_SUBSTEPS          = N_SUBSTEPS
_jw.FULL_FALL_STEPS     = FULL_FALL_STEPS
_jw.FULL_FALL_KE_THRESH = FULL_FALL_KE_THRESH
_jw.FULL_FALL_FRAME_SKIP= FULL_FALL_FRAME_SKIP
_jw.DISP_THRESH_FRAC    = DISP_THRESH_FRAC
_jw.KE_THRESH           = KE_THRESH
_jw.ANGLE_THRESH_DEG    = ANGLE_THRESH_DEG

print("All physics functions imported from jenga_worker")
print(f"  Classes: RigidBody, World, Contact")
print(f"  Functions: build_tower, run_episode, composite_collapse_check, ziglar_block_analysis, …")

# Smoke test
w_test, b_test = build_tower(n_layers=3, mu_static=0.4)
print(f"  Smoke test: 3-layer tower = {len(b_test)} blocks  ✓")
del w_test, b_test


## 2. Rendering and Video Capture

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

LAYER_COLORS = ["#c97a4a","#d99a63","#e0b27e"]
FLOOR_COLOR  = "#8B7355"

def _box_faces(corners):
    idx=[[0,1,3,2],[4,6,7,5],[0,2,6,4],[1,5,7,3],[0,4,5,1],[2,3,7,6]]
    return [[corners[i] for i in f] for f in idx]

def _corners_from_snap(blk):
    R=quat_to_rotmat(np.array(blk["quat"]))
    return box_corners(np.array(blk["position"]),np.array(blk["half_extents"]),R)

def _render_snap_array(snapshot, floor_extent=0.14, elev=22, azim=35):
    fig=plt.figure(figsize=(5,5),dpi=90)
    ax=fig.add_subplot(111,projection="3d")
    for blk in snapshot:
        corners=_corners_from_snap(blk); faces=_box_faces(corners)
        ax.add_collection3d(Poly3DCollection(faces,facecolor=LAYER_COLORS[blk["slot"]%3],
            edgecolor="#3a2a1a",linewidths=0.3,alpha=0.95))
    ax.set_xlim(-floor_extent,floor_extent); ax.set_ylim(-floor_extent,floor_extent)
    zs=[b["position"][2]+b["half_extents"][2] for b in snapshot] or [0.1]
    mz=max(max(zs)*1.1,0.05); ax.set_zlim(0,mz)
    xx,yy=np.meshgrid([-floor_extent,floor_extent],[-floor_extent,floor_extent])
    ax.plot_surface(xx,yy,np.zeros_like(xx),alpha=0.12,color=FLOOR_COLOR)
    ax.view_init(elev=elev,azim=azim); ax.set_box_aspect((1,1,mz/floor_extent))
    ax.set_axis_off(); fig.tight_layout(pad=0.1); fig.canvas.draw()
    img=np.asarray(fig.canvas.buffer_rgba())[:,:,:3]; plt.close(fig)
    return img

print("render OK")


## 3. Run 150 × 3 Friction Levels

Spawn workers import `jenga_worker.run_episode` directly — bypassing
the Jupyter namespace. This resolves `AttributeError: Can't get attribute 'run_episode'`.


In [ ]:
import concurrent.futures, multiprocessing as _mp, time as _time

# The function sent to the pool MUST be importable by workers.
# run_episode is in jenga_worker.py → workers can import it with spawn.
from jenga_worker import run_episode as _worker_fn

def _get_executor(n):
    """spawn: clean processes, no CUDA inheritance or Jupyter globals."""
    for method in ("spawn", "forkserver"):
        try:
            ctx = _mp.get_context(method)
            ex  = concurrent.futures.ProcessPoolExecutor(max_workers=n, mp_context=ctx)
            print(f"  ProcessPoolExecutor with start_method='{method}' ({n} workers)")
            return ex
        except Exception as e:
            print(f"  {method} not available: {e}")
    return None

all_results = {}

for mu_name, mu_val in FRICTION_LEVELS.items():
    print(f"\n{'='*60}")
    print(f"  µ={mu_val:.2f} ({mu_name.upper()})")
    fm=FAPP_MIN_FACTOR*mu_val*BLOCK_MASS_KG*G_ACCEL*1000
    ft=FAPP_TORQUE_FACTOR*mu_val*BLOCK_MASS_KG*G_ACCEL*1000
    print(f"  Fapp_min={fm:.1f} mN  Fapp_torque={ft:.1f} mN  (Ziglar Eq.4 / Eq.8)")
    print(f"{'='*60}")

    # Sync simulation constants to worker module before spawn
    import jenga_worker as _jw2
    _jw2.SOLVER_ITERS        = SOLVER_ITERS
    _jw2.N_SUBSTEPS          = N_SUBSTEPS
    _jw2.FULL_FALL_STEPS     = FULL_FALL_STEPS
    _jw2.FULL_FALL_KE_THRESH = FULL_FALL_KE_THRESH
    _jw2.FULL_FALL_FRAME_SKIP= FULL_FALL_FRAME_SKIP
    _jw2.DISP_THRESH_FRAC    = DISP_THRESH_FRAC
    _jw2.KE_THRESH           = KE_THRESH
    _jw2.ANGLE_THRESH_DEG    = ANGLE_THRESH_DEG
    # Rewrite the file with updated values (workers re-import it)
    import inspect
    src = open(os.path.join(os.getcwd(),"jenga_worker.py")).read()
    for _const,_val in [("SOLVER_ITERS",SOLVER_ITERS),("N_SUBSTEPS",N_SUBSTEPS),
                         ("FULL_FALL_STEPS",FULL_FALL_STEPS),
                         ("FULL_FALL_KE_THRESH",FULL_FALL_KE_THRESH),
                         ("FULL_FALL_FRAME_SKIP",FULL_FALL_FRAME_SKIP)]:
        import re
        src = re.sub(rf"^{_const}\s*=.*$", f"{_const} = {_val}", src, flags=re.MULTILINE)
    with open(os.path.join(os.getcwd(),"jenga_worker.py"),"w") as _ff:
        _ff.write(src)

    rng_m = np.random.default_rng(SEED + hash(mu_name)%1000)
    ep_args = [
        (eid, N_LAYERS, SETTLE, SOLVER_ITERS, N_SUBSTEPS,
         int(rng_m.integers(0,2**31)), "data", mu_name, mu_val)
        for eid in range(N_EXPERIMENTS)
    ]

    t0=_time.time(); raw=[None]*N_EXPERIMENTS; nd=0; nc=0

    def _report():
        el=_time.time()-t0; eta=(N_EXPERIMENTS-nd)/max(nd/el,1e-6)
        print(f"  {nd:3d}/{N_EXPERIMENTS}  collapsed={nc}  {el:.0f}s  ETA {eta:.0f}s")

    done_parallel = False
    if N_WORKERS > 1:
        ex = _get_executor(N_WORKERS)
        if ex is not None:
            done_parallel = True
            try:
                with ex as pool:
                    # Use _worker_fn (imported from jenga_worker) — pickleable
                    futs = {pool.submit(_worker_fn, a): a[0] for a in ep_args}
                    for fut in concurrent.futures.as_completed(futs):
                        eid = futs[fut]
                        try:    res=fut.result()
                        except Exception as e: print(f"  ⚠ exp {eid}: {e}"); res=None
                        raw[eid]=res; nd+=1
                        if res and res["collapsed"]: nc+=1
                        if nd%max(1,N_EXPERIMENTS//10)==0 or nd==N_EXPERIMENTS: _report()
            except Exception as pe:
                print(f"  Pool error: {pe} — sequential fallback"); done_parallel=False

    if not done_parallel:
        print("  Sequential mode (Numba JIT warm-up on ep 0)")
        for a in ep_args:
            res,eid=_run_and_track(a); raw[eid]=res; nd+=1
            if res and res["collapsed"]: nc+=1
            if nd==1: print(f"  ep 0 done — JIT cached")
            if nd%max(1,N_EXPERIMENTS//10)==0 or nd==N_EXPERIMENTS: _report()

    results=[r for r in raw if r is not None]
    tt=_time.time()-t0; all_results[mu_name]=results
    avg_r=np.mean([r["n_rounds"] for r in results]) if results else float("nan")
    print(f"\n✓ {len(results)}/{N_EXPERIMENTS} ep  {tt:.1f}s ({tt/max(len(results),1):.1f}s/ep)")
    print(f"  Collapsed: {nc} ({100*nc/max(len(results),1):.1f}%)  Avg rounds: {avg_r:.1f}")
    slim=[{k:v for k,v in r.items() if k not in("rounds","frame_snapshots","fall_frames")} for r in results]
    with open(f"data/experiments/summary_{mu_name}.json","w") as f: json.dump(slim,f,indent=2)

print("\n"+"="*60+"\nGLOBAL SUMMARY\n"+"="*60)
for mn,res in all_results.items():
    nc2=sum(1 for r in res if r["collapsed"])
    avg=np.mean([r["n_rounds"] for r in res]) if res else float("nan")
    print(f"  µ={FRICTION_LEVELS[mn]:.2f} ({mn:7s}): {nc2:3d}/{len(res)} collapsed  avg_rounds={avg:.1f}")


## 4. Videos — Collapse + Full Floor Fall

In [ ]:
import imageio.v2 as _imageio, time as _tv

def render_video(result, fps=15):
    eid=result["exp_id"]; mu=result["mu_level"]
    path=f"data/videos/{mu}_exp_{eid:04d}.mp4"
    frames=[]
    for rf in result.get("frame_snapshots",[]): frames.extend(rf)
    frames.extend(result.get("fall_frames",[]))
    if not frames: return None
    w=_imageio.get_writer(path,fps=fps,macro_block_size=None)
    try:
        for snap in frames: w.append_data(_render_snap_array(snap))
    finally: w.close()
    return path

print("Rendering videos…")
t0v=_tv.time(); vpaths={}
for mu_name,results in all_results.items():
    vp=[]
    for i,res in enumerate(results):
        p=render_video(res)
        if p: vp.append(p)
        if (i+1)%30==0 or i==len(results)-1:
            print(f"  {mu_name}: {i+1}/{len(results)}  ({_tv.time()-t0v:.0f}s)")
    vpaths[mu_name]=vp

total_v=sum(len(v) for v in vpaths.values())
print(f"\n✓ {total_v} videos in {_tv.time()-t0v:.1f}s")

from IPython.display import Video, display
for mu_name,results in all_results.items():
    col=[r for r in results if r["collapsed"]]
    if col:
        r0=col[0]; p=f"data/videos/{mu_name}_exp_{r0['exp_id']:04d}.mp4"
        if os.path.exists(p):
            print(f"\n  µ={FRICTION_LEVELS[mu_name]:.2f} ({mu_name}): exp {r0['exp_id']} — {r0['n_rounds']} rounds")
            display(Video(p,embed=True,width=380))

# Final-state images
print("\nGenerating final-state images…")
from mpl_toolkits.mplot3d.art3d import Poly3DCollection as _P3D
for mu_name,results in all_results.items():
    for res in results:
        eid=res["exp_id"]; last=res["rounds"][-1] if res["rounds"] else None
        if not last: continue
        snap=[dict(id=bid,position=pos,quat=last["quats"][bid],
                    half_extents=[_jw.L_BLOCK/2,_jw.W_BLOCK/2,_jw.H_BLOCK/2],
                    slot=int(bid.split("_")[-1]))
              for bid,pos in last["positions"].items()]
        fig=plt.figure(figsize=(4,4),dpi=80); ax=fig.add_subplot(111,projection="3d")
        for blk in snap:
            corners=_corners_from_snap(blk); faces=_box_faces(corners)
            ax.add_collection3d(_P3D(faces,facecolor=LAYER_COLORS[blk["slot"]%3],
                                      edgecolor="#3a2a1a",linewidths=0.3,alpha=0.95))
        ax.set_xlim(-0.14,0.14); ax.set_ylim(-0.14,0.14)
        zs=[b["position"][2]+b["half_extents"][2] for b in snap] or [0.1]
        ax.set_zlim(0,max(max(zs)*1.1,0.05)); ax.view_init(22,35)
        ax.set_box_aspect((1,1,1)); ax.set_axis_off()
        ax.set_title(f"µ={FRICTION_LEVELS[mu_name]:.2f} exp{eid} {'COL' if res['collapsed'] else 'ok'}",fontsize=7)
        fig.tight_layout(pad=0.1)
        fig.savefig(f"data/frames/{mu_name}_exp_{eid:04d}_final.png",dpi=80)
        plt.close(fig)
print("✓ Final-state images saved")


## 5. Dashboard — Analysis by Friction Level + Ziglar

In [ ]:
from IPython.display import Image, display

fig=plt.figure(figsize=(18,16))
fig.suptitle("Jenga Dashboard — Ziglar (2006) × 3 Friction Levels",
             fontsize=14,fontweight="bold")
cmu={"low":"#3498DB","nominal":"#E67E22","high":"#27AE60"}
mnames=list(FRICTION_LEVELS.keys())

ax1=fig.add_subplot(3,4,1)
cr=[sum(1 for r in all_results[m] if r["collapsed"])/max(len(all_results[m]),1) for m in mnames]
bars=ax1.bar(mnames,[r*100 for r in cr],color=[cmu[m] for m in mnames],edgecolor="k",lw=0.7)
for b,r in zip(bars,cr): ax1.text(b.get_x()+b.get_width()/2,b.get_height()+1,f"{r*100:.1f}%",ha="center",va="bottom",fontsize=9,fontweight="bold")
ax1.set_ylabel("Collapse rate (%)"); ax1.set_title("Collapse by µ"); ax1.set_ylim(0,110)

ax2=fig.add_subplot(3,4,2)
for mn in mnames: ax2.hist([r["n_rounds"] for r in all_results[mn]],bins=15,alpha=0.6,label=f"µ={FRICTION_LEVELS[mn]:.2f}",color=cmu[mn])
ax2.set_xlabel("Rounds"); ax2.set_ylabel("Episodes"); ax2.set_title("Round distribution"); ax2.legend(fontsize=7)

ax3=fig.add_subplot(3,4,3)
fmin=[FAPP_MIN_FACTOR*v*BLOCK_MASS_KG*G_ACCEL*1000 for v in FRICTION_LEVELS.values()]
ftor=[FAPP_TORQUE_FACTOR*v*BLOCK_MASS_KG*G_ACCEL*1000 for v in FRICTION_LEVELS.values()]
x=np.arange(len(mnames))
ax3.bar(x-0.2,fmin,0.35,color="#3498DB",label="Fapp_min (Eq.4)")
ax3.bar(x+0.2,ftor,0.35,color="#E74C3C",label="Fapp_torque (Eq.8)")
ax3.set_xticks(x); ax3.set_xticklabels(mnames)
ax3.set_ylabel("Force (mN)"); ax3.set_title("Ziglar thresholds"); ax3.legend(fontsize=8)
for xi,fm,ft in zip(x,fmin,ftor):
    ax3.text(xi-0.2,fm+0.2,f"{fm:.0f}",ha="center",va="bottom",fontsize=7)
    ax3.text(xi+0.2,ft+0.2,f"{ft:.0f}",ha="center",va="bottom",fontsize=7)

ax4=fig.add_subplot(3,4,4)
for mn in mnames:
    margins=[rd["analytic_margin"] for r in all_results[mn] for rd in r["rounds"]]
    ax4.hist(margins,bins=40,alpha=0.6,label=f"µ={FRICTION_LEVELS[mn]:.2f}",color=cmu[mn],density=True)
ax4.axvline(0,color="red",ls="--",lw=1.5,label="stability limit")
ax4.set_xlabel("Analytical margin"); ax4.set_ylabel("Density")
ax4.set_title("COM margin vs support"); ax4.legend(fontsize=7)

ax5=fig.add_subplot(3,4,5)
risk_tot={"low":0,"medium":0,"high":0}
for mn,res in all_results.items():
    for r in res:
        for rd in r["rounds"]: risk_tot[rd.get("ziglar_risk","low")]=risk_tot.get(rd.get("ziglar_risk","low"),0)+1
tot_r=sum(risk_tot.values())
ax5.pie([risk_tot[k]/max(tot_r,1)*100 for k in ["low","medium","high"]],
        labels=["low","medium","high"],autopct="%1.1f%%",
        colors=["#27AE60","#E67E22","#E74C3C"],startangle=90)
ax5.set_title("Ziglar risk (all moves)")

ax6=fig.add_subplot(3,4,6)
for mi,mn in enumerate(mnames):
    tc=sum(1 for r in all_results[mn] for rd in r["rounds"] if rd.get("ziglar_can_torque"))
    ntc=sum(1 for r in all_results[mn] for rd in r["rounds"] if not rd.get("ziglar_can_torque"))
    tt=tc+ntc
    ax6.bar([mi-0.2],[tc/max(tt,1)*100],0.35,color="#E74C3C",label="With torque" if mi==0 else "")
    ax6.bar([mi+0.2],[ntc/max(tt,1)*100],0.35,color="#3498DB",label="Without torque" if mi==0 else "")
ax6.set_xticks(range(len(mnames))); ax6.set_xticklabels(mnames)
ax6.set_ylabel("% moves"); ax6.set_title("Moves with/without torque"); ax6.legend(fontsize=8)

ax7=fig.add_subplot(3,4,7)
for mn in mnames:
    ke=[rd["kinetic_energy"] for r in all_results[mn] for rd in r["rounds"] if rd["kinetic_energy"]>0]
    if ke: ax7.hist(np.log10(np.array(ke)+1e-12),bins=40,alpha=0.6,label=f"µ={FRICTION_LEVELS[mn]:.2f}",color=cmu[mn])
ax7.set_xlabel("log₁₀(KE)"); ax7.set_ylabel("Count"); ax7.set_title("Kinetic energy"); ax7.legend(fontsize=7)

ax8=fig.add_subplot(3,4,8)
for mn in mnames:
    for r in all_results[mn]:
        for rd in r["rounds"]:
            col="#E74C3C" if rd["collapsed"] else "#27AE60"
            ax8.scatter(rd["analytic_margin"],rd["max_displacement"]*100,c=col,alpha=0.15,s=4)
ax8.axvline(0,color="gray",ls="--",lw=1)
ax8.axhline(_jw.L_BLOCK*_jw.DISP_THRESH_FRAC*100,color="orange",ls="--",lw=1,label="threshold")
ax8.set_xlabel("Analytical margin"); ax8.set_ylabel("Max disp. (cm)")
ax8.set_title("Margin vs Displacement"); ax8.legend(fontsize=7)

ax9=fig.add_subplot(3,4,9)
for mn in mnames:
    lc=np.zeros(N_LAYERS,int)
    for r in all_results[mn]:
        for rd in r["rounds"]: lc[rd["removed_layer"]]+=1
    ax9.plot(range(N_LAYERS),lc,"o-",color=cmu[mn],label=f"µ={FRICTION_LEVELS[mn]:.2f}",lw=1.5,ms=4)
ax9.set_xlabel("Layer (0=base)"); ax9.set_ylabel("Times removed")
ax9.set_title("Removal frequency by layer"); ax9.legend(fontsize=7)

ax10=fig.add_subplot(3,4,10)
mtypes=["center_xaxis","side_yaxis","side_xaxis"]; mcols=["#9B59B6","#3498DB","#E74C3C"]
for mi2,mtype in enumerate(mtypes):
    rates=[]
    for mn in mnames:
        mc=sum(1 for r in all_results[mn] for rd in r["rounds"] if rd.get("ziglar_move_type")==mtype and rd["collapsed"])
        mt=sum(1 for r in all_results[mn] for rd in r["rounds"] if rd.get("ziglar_move_type")==mtype)
        rates.append(mc/max(mt,1)*100)
    ax10.bar(np.arange(len(mnames))+mi2*0.25,rates,0.22,color=mcols[mi2],label=mtype,alpha=0.85)
ax10.set_xticks(np.arange(len(mnames))+0.25); ax10.set_xticklabels(mnames)
ax10.set_ylabel("Collapse rate (%)"); ax10.set_title("Collapse by Ziglar type"); ax10.legend(fontsize=7)

ax11=fig.add_subplot(3,4,11)
for mn in mnames:
    col_r=[r for r in all_results[mn] if r["collapsed"]]
    if col_r:
        fl=[r.get("n_blocks_on_floor",0) for r in col_r]
        ax11.hist(fl,bins=max(5,1),alpha=0.7,label=f"µ={FRICTION_LEVELS[mn]:.2f}",color=cmu[mn])
ax11.set_xlabel("Blocks on floor"); ax11.set_ylabel("Episodes")
ax11.set_title("Blocks on floor after collapse"); ax11.legend(fontsize=7)

ax12=fig.add_subplot(3,4,12)
for mn in mnames:
    angles=[rd["max_angle_deg"] for r in all_results[mn] for rd in r["rounds"]]
    ax12.hist(angles,bins=30,alpha=0.6,label=f"µ={FRICTION_LEVELS[mn]:.2f}",color=cmu[mn],density=True)
ax12.axvline(ANGLE_THRESH_DEG,color="red",ls="--",lw=1.5,label=f"threshold {ANGLE_THRESH_DEG}°")
ax12.set_xlabel("Max angle (°)"); ax12.set_ylabel("Density")
ax12.set_title("Maximum block angle"); ax12.legend(fontsize=7)

fig.tight_layout(rect=[0,0,1,0.96])
fig.savefig("data/dashboard/ziglar_friction_dashboard.png",dpi=120,bbox_inches="tight")
plt.close(fig)
display(Image("data/dashboard/ziglar_friction_dashboard.png"))
print("✓ Dashboard saved")


## 6. Multi-task CNN (GPU if available)

In [ ]:
TORCH_AVAILABLE = False
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"PyTorch {torch.__version__}  device={DEVICE}" +
          (f"  ({torch.cuda.get_device_name(0)})" if DEVICE.type=="cuda" else ""))
except ImportError as e:
    print(f"PyTorch not available ({e})")

if TORCH_AVAILABLE:
    class JengaDS(Dataset):
        def __init__(self, recs, sz=IMG_SIZE, aug=True):
            self.recs=recs; self.sz=sz; self.aug=aug
            self.mu_map={n:i for i,n in enumerate(FRICTION_LEVELS)}
        def __len__(self): return len(self.recs)
        def __getitem__(self, idx):
            r=self.recs[idx]; mn=r.get("mu_level","nominal")
            p=f"data/frames/{mn}_exp_{r['exp_id']:04d}_final.png"
            from PIL import Image as _PIL
            try:
                img=_PIL.open(p).convert("L").resize((self.sz,self.sz))
                arr=torch.tensor(np.array(img,np.float32)/255.).unsqueeze(0)
            except: arr=torch.zeros(1,self.sz,self.sz)
            if self.aug:
                k=np.random.randint(0,4); arr=torch.rot90(arr,k,dims=[1,2])
                if np.random.rand()>.5: arr=torch.flip(arr,dims=[2])
            mu_oh=torch.zeros(3); mu_oh[self.mu_map.get(mn,1)]=1.
            last=r["rounds"][-1] if r["rounds"] else {}
            locs=torch.zeros(N_LAYERS*3)
            for bid in r.get("removed_order",[]):
                try:
                    l=int(bid[1:3]); s=int(bid.split("_")[-1]); fi=l*3+s
                    if fi<N_LAYERS*3: locs[fi]=1.
                except: pass
            return dict(image=arr,mu=mu_oh,
                        num_removed=torch.tensor(float(r["n_rounds"])),
                        locs=locs,
                        imbalance=torch.tensor(float(-last.get("analytic_margin",0.))),
                        torque_risk=torch.tensor(float(sum(1 for rd in r["rounds"] if rd.get("ziglar_can_torque")))))

    class JengaCNN(nn.Module):
        def __init__(self, n_slots=None, emb=128):
            super().__init__()
            if n_slots is None: n_slots=N_LAYERS*3
            try:
                from torchvision import models
                self.bb=models.resnet18(weights="IMAGENET1K_V1")
                oc=self.bb.conv1
                self.bb.conv1=nn.Conv2d(1,64,7,stride=2,padding=3,bias=False)
                with torch.no_grad(): self.bb.conv1.weight=nn.Parameter(oc.weight.mean(1,keepdim=True))
                fd=self.bb.fc.in_features; self.bb.fc=nn.Linear(fd,emb); self.has_bb=True
            except ImportError:
                self.has_bb=False
                self.bb=nn.Sequential(
                    nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                    nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                    nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(4),
                    nn.Flatten(),nn.Linear(64*16,emb))
            self.mu_proj=nn.Linear(3,16)
            self.fuse=nn.Linear(emb+16,emb)
            self.bn=nn.BatchNorm1d(emb); self.drop=nn.Dropout(0.3)
            self.h_num=nn.Linear(emb,1); self.h_loc=nn.Linear(emb,n_slots)
            self.h_imb=nn.Linear(emb,1); self.h_tor=nn.Linear(emb,1)
        def forward(self,x,mu):
            z=F.relu(self.bb(x)); mf=F.relu(self.mu_proj(mu))
            z=F.relu(self.fuse(torch.cat([z,mf],1))); z=self.drop(self.bn(z))
            return dict(num_removed=self.h_num(z).squeeze(-1),
                        locs_logits=self.h_loc(z),
                        imbalance=self.h_imb(z).squeeze(-1),
                        torque_risk=self.h_tor(z).squeeze(-1),embedding=z)

    print("CNN defined (ResNet-18 + friction_onehot + torque_risk head)")


## 7. Training

In [ ]:
from PIL import Image as _PIL

recs=[]
for mn,res in all_results.items():
    for r in res:
        p=f"data/frames/{mn}_exp_{r['exp_id']:04d}_final.png"
        if os.path.exists(p) and r["rounds"]: recs.append(r)
_per_mu = {mu_k: sum(1 for r in recs if r.get("mu_level")==mu_k) for mu_k in FRICTION_LEVELS}
print(f"Samples: {len(recs)}  ({_per_mu})")

if TORCH_AVAILABLE and len(recs)>=10:
    np.random.seed(SEED); idx=np.random.permutation(len(recs)); sp=int(.8*len(recs))
    tr=[recs[i] for i in idx[:sp]]; vr=[recs[i] for i in idx[sp:]]
    trd=JengaDS(tr,IMG_SIZE,True); vd=JengaDS(vr,IMG_SIZE,False)
    trl=DataLoader(trd,BATCH_SIZE,shuffle=True,num_workers=0)
    vll=DataLoader(vd, BATCH_SIZE,shuffle=False,num_workers=0)
    model=JengaCNN(N_LAYERS*3); model.to(DEVICE)
    print(f"Params: {sum(p.numel() for p in model.parameters()):,}  device={DEVICE}")
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,EPOCHS,eta_min=LR/20)
    scaler=torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))
    for ep in range(EPOCHS):
        model.train(); el=0.
        for b in trl:
            imgs=b["image"].to(DEVICE); mu=b["mu"].to(DEVICE)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=="cuda")):
                p2=model(imgs,mu)
                l=(0.4*F.mse_loss(p2["num_removed"],b["num_removed"].float().to(DEVICE))+
                   0.8*F.binary_cross_entropy_with_logits(p2["locs_logits"],b["locs"].float().to(DEVICE))+
                   0.4*F.mse_loss(p2["imbalance"],b["imbalance"].float().to(DEVICE))+
                   0.4*F.mse_loss(p2["torque_risk"],b["torque_risk"].float().to(DEVICE)))
            scaler.scale(l).backward(); scaler.step(opt); scaler.update(); el+=l.item()
        sched.step()
        if ep%5==0 or ep==EPOCHS-1:
            print(f"  ep {ep:3d}  train_loss={el/max(len(trl),1):.4f}  lr={sched.get_last_lr()[0]:.2e}")
    torch.save({"model_state":model.state_dict(),"n_slots":N_LAYERS*3,"img_size":IMG_SIZE},
               "models/jenga_friction_cnn.pt")
    print("Model saved to models/jenga_friction_cnn.pt")
else:
    print("sklearn fallback (PyTorch not available or dataset too small)")
    from sklearn.linear_model import Ridge; from sklearn.preprocessing import StandardScaler
    def _feat(r,sz=32):
        mn=r.get("mu_level","nominal")
        p=f"data/frames/{mn}_exp_{r['exp_id']:04d}_final.png"
        img=_PIL.open(p).convert("L").resize((sz,sz))
        pix=np.array(img,np.float32).flatten()/255.
        oh=np.zeros(3); oh[list(FRICTION_LEVELS).index(mn)]=1.
        return np.concatenate([pix,oh])
    X=np.array([_feat(r) for r in recs])
    yn=np.array([r["n_rounds"] for r in recs],float)
    sc=StandardScaler().fit(X); Xs=sc.transform(X)
    rn=Ridge(1.).fit(Xs,yn)
    print(f"  MAE num_removed={np.mean(np.abs(rn.predict(Xs)-yn)):.2f}")
    with open("models/sklearn_proxy.pkl","wb") as f: pickle.dump(dict(rn=rn,sc=sc,recs=recs),f)


## 8. Final Summary

In [ ]:
print("="*70)
print("JENGA PHYSICS SIMULATOR — FINAL SUMMARY")
print("Paper: Ziglar (2006) Analysis of Mechanics in Jenga, CMU")
print("="*70)
total_ep=sum(len(r) for r in all_results.values())
total_col=sum(sum(1 for r in res if r["collapsed"]) for res in all_results.values())
total_v=len(glob.glob("data/videos/*.mp4"))
print(f"\nExperiments: {total_ep} total ({N_EXPERIMENTS} × {len(FRICTION_LEVELS)} levels)")
print(f"Collapsed: {total_col} ({100*total_col/max(total_ep,1):.1f}%)")
print(f"Videos: {total_v}  Images: {len(glob.glob('data/frames/*_final.png'))}")
print(f"\nBy friction level:")
for mn,mv in FRICTION_LEVELS.items():
    res=all_results[mn]; nc=sum(1 for r in res if r["collapsed"])
    avg=np.mean([r["n_rounds"] for r in res]) if res else float("nan")
    fm=FAPP_MIN_FACTOR*mv*BLOCK_MASS_KG*G_ACCEL*1000
    ft=FAPP_TORQUE_FACTOR*mv*BLOCK_MASS_KG*G_ACCEL*1000
    tc=sum(1 for r in res for rd in r["rounds"] if rd.get("ziglar_can_torque"))
    tt=sum(r["n_rounds"] for r in res)
    print(f"  µ={mv:.2f} ({mn}): {nc}/{len(res)} collapsed  avg={avg:.1f}  "
          f"Fapp={fm:.0f}/{ft:.0f}mN  torque={100*tc/max(tt,1):.0f}%")
print(f"\nHardware: GPU={'✓ '+gpu_name if GPU_AVAILABLE else '✗ CPU'}  "
      f"Numba={'✓' if NUMBA_AVAILABLE else '✗'}  "
      f"PyTorch={'✓ '+str(DEVICE) if TORCH_AVAILABLE else '✗'}")
print(f"\nMain file: jenga_worker.py (physics module for spawn workers)")
print(f"Dashboard: data/dashboard/ziglar_friction_dashboard.png")
print("="*70)


## 9. Inference + Reverse Reconstruction Video

> **How to use:** set `TEST_IMAGE_PATH` to any PNG with fallen blocks and `TEST_MU_LEVEL` to the estimated friction level. The cell produces:
> 1. **Inference panel** — input image · removal heat map · model predictions
> 2. **Video `reconstruction_*.mp4`** — forward sequence to collapse → prediction overlay pause → reverse rebuilding the tower

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ## 9. Inference + Reverse Reconstruction Video
#
# How to use:
#   1. Set TEST_IMAGE_PATH to your PNG image of fallen blocks.
#      Can be any image from data/frames/ or your own image.
#   2. Set TEST_MU_LEVEL to the friction level you think the game had
#      ("low", "nominal" or "high"). If unsure, leave "nominal".
#   3. Run the cell. The model predicts:
#        - how many blocks were removed before collapse
#        - which tower positions were removed
#        - accumulated torque risk (Ziglar)
#        - estimated COM imbalance
#   4. Generates data/videos/reconstruction_<name>.mp4 with:
#        → the FORWARD sequence (base → collapse), then
#        → the REVERSE sequence (collapse → reconstructed base)
# ═══════════════════════════════════════════════════════════════════════════════

# ── INPUT PARAMETERS ─────────────────────────────────────────────────────────
TEST_IMAGE_PATH = "data/frames/nominal_exp_0000_final.png"  # ← change here
TEST_MU_LEVEL   = "nominal"   # "low" | "nominal" | "high"

# ── 0. Rendering helpers with overlay ────────────────────────────────────────
import numpy as np, os, json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from PIL import Image as _PIL
import imageio.v2 as _imageio

_RISK_COLORS = {"low": "#27AE60", "medium": "#E67E22", "high": "#E74C3C"}

def _render_snap_overlay(snapshot, title="", subtitle="", floor_extent=0.14,
                          elev=22, azim=35, highlight_slots=None, dpi=110):
    """
    Renders a snapshot with title and subtitle overlay.
    highlight_slots: list of (layer, slot) to highlight in bright red.
    """
    fig = plt.figure(figsize=(5, 5.6), dpi=dpi)
    ax3d = fig.add_axes([0, 0.12, 1, 0.88], projection="3d")

    hi = set(highlight_slots or [])
    for blk in snapshot:
        corners = _corners_from_snap(blk)
        faces   = _box_faces(corners)
        try:
            layer = int(blk["id"][1:3]); slot = int(blk["id"].split("_")[-1])
        except Exception:
            layer, slot = 0, blk["slot"]
        is_hi = (layer, slot) in hi
        color  = "#FF2222" if is_hi else LAYER_COLORS[blk["slot"] % 3]
        alpha  = 1.0 if is_hi else 0.92
        ax3d.add_collection3d(Poly3DCollection(
            faces, facecolor=color, edgecolor="#3a2a1a",
            linewidths=0.5 if is_hi else 0.25, alpha=alpha))

    ax3d.set_xlim(-floor_extent, floor_extent)
    ax3d.set_ylim(-floor_extent, floor_extent)
    zs  = [b["position"][2] + b["half_extents"][2] for b in snapshot] or [0.1]
    mz  = max(max(zs) * 1.15, 0.05)
    ax3d.set_zlim(0, mz)
    xx, yy = np.meshgrid([-floor_extent, floor_extent],
                          [-floor_extent, floor_extent])
    ax3d.plot_surface(xx, yy, np.zeros_like(xx), alpha=0.13, color=FLOOR_COLOR)
    ax3d.view_init(elev=elev, azim=azim)
    ax3d.set_box_aspect((1, 1, mz / floor_extent))
    ax3d.set_axis_off()

    ax2d = fig.add_axes([0, 0, 1, 0.13])
    ax2d.set_axis_off()
    ax2d.text(0.5, 0.72, title,    ha="center", va="center",
              fontsize=10, fontweight="bold", transform=ax2d.transAxes, color="#1a1a2e")
    ax2d.text(0.5, 0.22, subtitle, ha="center", va="center",
              fontsize=7.5, transform=ax2d.transAxes, color="#555555")
    fig.patch.set_facecolor("#F8F4EE")
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    plt.close(fig)
    return img


def _render_grid_panel(snap_list, titles, ncols=4, dpi=90):
    """Grid of snapshots for the inference panel."""
    nrows = (len(snap_list) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(ncols * 3, nrows * 3.2),
                              subplot_kw={"projection": "3d"})
    axes = np.array(axes).flatten()
    for i, (snap, ttl) in enumerate(zip(snap_list, titles)):
        ax = axes[i]
        for blk in snap:
            corners = _corners_from_snap(blk)
            ax.add_collection3d(Poly3DCollection(
                _box_faces(corners),
                facecolor=LAYER_COLORS[blk["slot"] % 3],
                edgecolor="#3a2a1a", linewidths=0.2, alpha=0.92))
        zs = [b["position"][2] + b["half_extents"][2] for b in snap] or [0.1]
        mz = max(max(zs) * 1.1, 0.05)
        ax.set_xlim(-0.14, 0.14); ax.set_ylim(-0.14, 0.14); ax.set_zlim(0, mz)
        ax.view_init(22, 35); ax.set_box_aspect((1, 1, 1)); ax.set_axis_off()
        ax.set_title(ttl, fontsize=7.5, pad=2)
    for ax in axes[len(snap_list):]:
        ax.set_visible(False)
    fig.tight_layout(pad=0.3)
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    plt.close(fig)
    return img


# ── 1. Load and pre-process the input image ──────────────────────────────────
print("─" * 60)
print(f"  Input image    : {TEST_IMAGE_PATH}")
print(f"  Friction level : {TEST_MU_LEVEL}  (µ={FRICTION_LEVELS[TEST_MU_LEVEL]:.2f})")
print("─" * 60)

if not os.path.exists(TEST_IMAGE_PATH):
    # Image not found — search for any final-state image
    candidates = sorted(glob.glob("data/frames/*_final.png"))
    if candidates:
        TEST_IMAGE_PATH = candidates[0]
        print(f"  ⚠ Image not found — using: {TEST_IMAGE_PATH}")
    else:
        raise FileNotFoundError(
            "No images in data/frames/. Run the simulation cells first.")

input_img_pil = _PIL.open(TEST_IMAGE_PATH).convert("L")
input_img_rgb = _PIL.open(TEST_IMAGE_PATH).convert("RGB")
print(f"  Image loaded: {input_img_rgb.size[0]}×{input_img_rgb.size[1]} px")


# ── 2. Model inference ───────────────────────────────────────────────────────
mu_idx  = list(FRICTION_LEVELS.keys()).index(TEST_MU_LEVEL)
mu_val  = FRICTION_LEVELS[TEST_MU_LEVEL]
n_slots = N_LAYERS * 3

pred_num_removed   = None
pred_locs_prob     = None
pred_imbalance     = None
pred_torque_risk   = None
used_model         = "none"
embedding          = None

if TORCH_AVAILABLE and os.path.exists("models/jenga_friction_cnn.pt"):
    ckpt = torch.load("models/jenga_friction_cnn.pt",
                      map_location=DEVICE, weights_only=False)
    _n_slots = ckpt.get("n_slots", n_slots)
    _img_sz  = ckpt.get("img_size", IMG_SIZE)

    infer_model = JengaCNN(n_slots=_n_slots)
    infer_model.load_state_dict(ckpt["model_state"])
    infer_model.to(DEVICE).eval()

    img_t = torch.tensor(
        np.array(input_img_pil.resize((_img_sz, _img_sz)), np.float32) / 255.
    ).unsqueeze(0).unsqueeze(0).to(DEVICE)          # (1,1,H,W)
    mu_oh = torch.zeros(1, 3).to(DEVICE)
    mu_oh[0, mu_idx] = 1.

    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            out = infer_model(img_t, mu_oh)

    pred_num_removed = float(out["num_removed"].cpu().item())
    pred_locs_prob   = torch.sigmoid(out["locs_logits"]).cpu().numpy().flatten()
    pred_imbalance   = float(out["imbalance"].cpu().item())
    pred_torque_risk = float(out["torque_risk"].cpu().item())
    embedding        = out["embedding"].cpu().numpy().flatten()
    used_model = f"JengaCNN (PyTorch {torch.__version__}, {DEVICE})"
    print(f"\n✓ Inference with JengaCNN")

elif os.path.exists("models/sklearn_proxy.pkl"):
    import pickle
    from sklearn.preprocessing import StandardScaler
    with open("models/sklearn_proxy.pkl", "rb") as f:
        pkg = pickle.load(f)
    sz = 32
    pix = np.array(input_img_pil.resize((sz, sz)), np.float32).flatten() / 255.
    oh  = np.zeros(3); oh[mu_idx] = 1.
    feat = np.concatenate([pix, oh]).reshape(1, -1)
    Xs   = pkg["sc"].transform(feat)
    pred_num_removed = float(pkg["rn"].predict(Xs)[0])
    pred_locs_prob   = np.zeros(n_slots)   # sklearn has no localisation head
    pred_imbalance   = float(pkg.get("ri", pkg["rn"]).predict(Xs)[0])                        if "ri" in pkg else 0.
    pred_torque_risk = 0.
    used_model = "sklearn Ridge (proxy)"
    print(f"\n✓ Inference with sklearn Ridge (fallback)")
else:
    # No model: heuristic estimate based on Ziglar static analysis
    print("\n⚠ No saved model found — using Ziglar heuristic estimate")
    pred_num_removed = N_LAYERS * 0.4
    pred_locs_prob   = np.random.rand(n_slots) * 0.3
    pred_imbalance   = 0.01
    pred_torque_risk = pred_num_removed * 0.3
    used_model = "Ziglar heuristic"

# ── 3. Decode predictions ────────────────────────────────────────────────────
n_removed_pred = max(1, int(round(pred_num_removed)))
fapp_min_pred  = FAPP_MIN_FACTOR    * mu_val * BLOCK_MASS_KG * G_ACCEL * 1000
fapp_tor_pred  = FAPP_TORQUE_FACTOR * mu_val * BLOCK_MASS_KG * G_ACCEL * 1000

# Top blocks predicted as removed (probability > threshold)
PROB_THRESH = 0.35
removed_slots = [(i // 3, i % 3) for i, p in enumerate(pred_locs_prob)
                 if p > PROB_THRESH]
# Sort by descending probability and take top n_removed_pred
top_removed = sorted(enumerate(pred_locs_prob), key=lambda x: -x[1])[:n_removed_pred]
removed_ids  = [f"L{i//3:02d}_{i%3}" for i, _ in top_removed]

torque_pct = min(100., pred_torque_risk / max(n_removed_pred, 1) * 100)
risk_label = ("high"   if torque_pct > 60 else
              "medium" if torque_pct > 30 else "low")

print(f"\n  Model used          : {used_model}")
print(f"  Blocks removed      : {n_removed_pred}  (pred={pred_num_removed:.2f})")
print(f"  Predicted blocks    : {removed_ids[:8]}{'…' if len(removed_ids)>8 else ''}")
print(f"  COM imbalance       : {pred_imbalance*1000:.2f} mm (estimated)")
print(f"  Torque risk         : {torque_pct:.1f}%  → {risk_label.upper()}")
print(f"  Fapp_min (Ziglar)   : {fapp_min_pred:.1f} mN  |  Fapp_torque: {fapp_tor_pred:.1f} mN")


# ── 4. Reconstruct the snapshot sequence ─────────────────────────────────────
# Find the most similar episode in all_results to use its real snapshots
# (If the image comes from the session, we can recover the original frames)

best_result = None
best_match_score = -1

_basename = os.path.splitext(os.path.basename(TEST_IMAGE_PATH))[0]
# Try to parse exp_id from filename: {mu}_exp_{eid:04d}_final
for mu_name, results in all_results.items():
    for r in results:
        expected_name = f"{mu_name}_exp_{r['exp_id']:04d}_final"
        if expected_name == _basename:
            best_result = r
            best_match_score = 999
            break
    if best_result: break

if best_result is None:
    # Search for collapsed episode with n_rounds closest to prediction
    for mu_name, results in all_results.items():
        for r in results:
            if r["collapsed"] and r["rounds"]:
                score = -abs(r["n_rounds"] - n_removed_pred)
                if score > best_match_score:
                    best_match_score = score
                    best_result = r

if best_result is None:
    # Fall back to episode with most rounds
    for mu_name, results in all_results.items():
        for r in results:
            if r["rounds"]:
                if best_result is None or r["n_rounds"] > best_result["n_rounds"]:
                    best_result = r

print(f"\n  Reference episode: µ={FRICTION_LEVELS.get(best_result['mu_level'],'?'):.2f} "
      f"exp={best_result['exp_id']} ({best_result['n_rounds']} rondas)")

# Collect all snapshots from the reference episode
all_snaps_forward = []   # (snapshot, label, round_idx, is_collapse_frame)
for ri, rd in enumerate(best_result["rounds"]):
    round_snaps = best_result["frame_snapshots"][ri] if ri < len(best_result["frame_snapshots"]) else []
    n_active = rd["n_active"]
    ziglar_r = rd.get("ziglar_risk", "low")
    for fi, snap in enumerate(round_snaps):
        label = (f"Round {ri+1} — removed {rd['removed_block']}\n"
                 f"Ziglar: {rd['ziglar_move_type']} | risk {ziglar_r.upper()} | "
                 f"{n_active} active blocks")
        all_snaps_forward.append((snap, label, ri, rd["collapsed"] and fi == len(round_snaps)-1))

# Add full floor-fall frames
fall_snaps = best_result.get("fall_frames", [])
for fi, snap in enumerate(fall_snaps):
    label = f"COLLAPSE — free fall {fi+1}/{len(fall_snaps)}\nBlocks on the floor…"
    all_snaps_forward.append((snap, label, -1, True))

print(f"  Frames available: {len(all_snaps_forward)} "
      f"({len(fall_snaps)} floor-fall frames)")


# ── 5. Generate combined video: forward sequence + reverse ───────────────────
os.makedirs("data/videos", exist_ok=True)
_stem = os.path.splitext(os.path.basename(TEST_IMAGE_PATH))[0]
out_video_path = f"data/videos/reconstruction_{_stem}.mp4"

# Predicted removed blocks to highlight in reverse
pred_highlight = set()
for bid in removed_ids:
    try:
        layer = int(bid[1:3]); slot = int(bid.split("_")[-1])
        pred_highlight.add((layer, slot))
    except Exception:
        pass

fps_forward  = 12
fps_reverse  = 10   # slightly slower reverse so it is easier to follow
pause_frames = 18   # frozen frames at the transition point

print(f"\n  Generando video: {out_video_path}")
print(f"  → Forward sequence : {len(all_snaps_forward)} frames @ {fps_forward} fps")
print(f"  → Transition pause : {pause_frames} frames")
print(f"  → Reverse sequence : {len(all_snaps_forward)} frames @ {fps_reverse} fps")

writer = _imageio.get_writer(out_video_path, fps=fps_forward, macro_block_size=None)

try:
    # ── PHASE 1: Forward sequence ─────────────────────────────────────────────
    for frame_i, (snap, label, ri, is_col) in enumerate(all_snaps_forward):
        parts = label.split("\n")
        title    = parts[0] if parts else label
        subtitle = parts[1] if len(parts) > 1 else ""
        # Highlight predicted blocks in forward frames
        img = _render_snap_overlay(snap, title=title, subtitle=subtitle,
                                    highlight_slots=list(pred_highlight) if is_col else None)
        writer.append_data(img)

    # ── PHASE 2: Frozen transition frame with prediction overlay ──────────────
    if all_snaps_forward:
        last_snap, _, _, _ = all_snaps_forward[-1]
        mu_display = FRICTION_LEVELS[TEST_MU_LEVEL]
        trans_title    = f"MODEL PREDICTION  ·  µ={mu_display:.2f} ({TEST_MU_LEVEL})"
        trans_subtitle = (f"{n_removed_pred} blocks removed  |  torque risk {risk_label.upper()}  "
                          f"|  imbalance {pred_imbalance*1000:.1f} mm")
        trans_img = _render_snap_overlay(
            last_snap, title=trans_title, subtitle=trans_subtitle,
            highlight_slots=list(pred_highlight))
        for _ in range(pause_frames):
            writer.append_data(trans_img)

    # ── PHASE 3: Reverse sequence ─────────────────────────────────────────────
    # Switch fps for the reverse segment
    writer.close()
    writer = _imageio.get_writer(out_video_path.replace(".mp4", "_tmp.mp4"),
                                  fps=fps_reverse, macro_block_size=None)

    # Reverse transition frame
    if all_snaps_forward:
        last_snap, _, _, _ = all_snaps_forward[-1]
        rev_title    = "◄◄  REVERSE RECONSTRUCTION"
        rev_subtitle = "Blocks returning to their original positions"
        rev_trans = _render_snap_overlay(last_snap, title=rev_title,
                                          subtitle=rev_subtitle,
                                          highlight_slots=list(pred_highlight))
        for _ in range(pause_frames // 2):
            writer.append_data(rev_trans)

    total_rev = len(all_snaps_forward)
    for frame_i, (snap, label, ri, is_col) in enumerate(reversed(all_snaps_forward)):
        parts = label.split("\n")
        # In reverse: label which round is being "undone"
        if ri >= 0:
            rev_rnd   = best_result["n_rounds"] - ri
            title_r   = f"◄ Undoing round {ri+1}  ({rev_rnd} of {best_result['n_rounds']})"
            sub_r     = parts[1] if len(parts) > 1 else ""
        else:
            title_r = "◄ Rewinding the fall…"
            sub_r   = ""
        # Highlight predicted last-to-fall blocks in red
        hi_rev = list(pred_highlight) if frame_i < len(fall_snaps) + 5 else None
        img = _render_snap_overlay(snap, title=title_r, subtitle=sub_r,
                                    highlight_slots=hi_rev)
        writer.append_data(img)

    # Final frame: fully reconstructed tower
    if best_result["frame_snapshots"] and best_result["frame_snapshots"][0]:
        first_snap = best_result["frame_snapshots"][0][0]
        end_title    = "✓ Tower reconstructed"
        end_subtitle = f"µ={FRICTION_LEVELS[TEST_MU_LEVEL]:.2f} | {N_LAYERS*3} blocks | Ziglar Fapp_min={fapp_min_pred:.0f} mN"
        end_img = _render_snap_overlay(first_snap, title=end_title, subtitle=end_subtitle)
        for _ in range(pause_frames):
            writer.append_data(end_img)

    writer.close()

    # Concatenate both segments with ffmpeg if available, otherwise use imageio
    import subprocess
    fwd_path = out_video_path
    rev_path = out_video_path.replace(".mp4", "_tmp.mp4")
    concat_path = out_video_path.replace(".mp4", "_concat.mp4")

    ffmpeg_ok = False
    try:
        list_file = out_video_path.replace(".mp4", "_list.txt")
        with open(list_file, "w") as lf:
            lf.write(f"file '{os.path.abspath(fwd_path)}'\n")
            lf.write(f"file '{os.path.abspath(rev_path)}'\n")
        res = subprocess.run(
            ["ffmpeg", "-y", "-f", "concat", "-safe", "0",
             "-i", list_file, "-c", "copy", concat_path],
            capture_output=True, timeout=60)
        if res.returncode == 0 and os.path.exists(concat_path):
            os.replace(concat_path, out_video_path)
            os.remove(rev_path)
            os.remove(list_file)
            ffmpeg_ok = True
            print(f"  ✓ Concatenated with ffmpeg")
    except Exception as _fe:
        pass

    if not ffmpeg_ok:
        # No ffmpeg: concatenate manually frame by frame
        print("  ffmpeg not available — concatenating with imageio")
        all_video_paths_combined = [fwd_path, rev_path]
        final_writer = _imageio.get_writer(out_video_path + ".combined.mp4",
                                            fps=fps_forward, macro_block_size=None)
        for vp in all_video_paths_combined:
            if os.path.exists(vp):
                reader = _imageio.get_reader(vp)
                for frame in reader:
                    final_writer.append_data(frame)
                reader.close()
        final_writer.close()
        os.replace(out_video_path + ".combined.mp4", out_video_path)
        if os.path.exists(rev_path): os.remove(rev_path)
        print(f"  ✓ Concatenated with imageio")

except Exception as _ve:
    try: writer.close()
    except: pass
    print(f"  ⚠ Error generating video: {_ve}")
    import traceback; traceback.print_exc()


# ── 6. Visual inference panel ────────────────────────────────────────────────
fig_inf, axes_inf = plt.subplots(1, 3, figsize=(15, 5))
fig_inf.suptitle(
    f"Jenga Model Inference  ·  µ={FRICTION_LEVELS[TEST_MU_LEVEL]:.2f} ({TEST_MU_LEVEL})",
    fontsize=13, fontweight="bold")

# Left panel: input image
axes_inf[0].imshow(input_img_rgb); axes_inf[0].set_axis_off()
axes_inf[0].set_title("Input image", fontsize=11)

# Centre panel: removal probability heat map
heat = pred_locs_prob.reshape(N_LAYERS, 3)
im   = axes_inf[1].imshow(heat, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1,
                            origin="lower")
axes_inf[1].set_xlabel("Slot (0=left, 1=centre, 2=right)")
axes_inf[1].set_ylabel("Layer (0=base)")
axes_inf[1].set_title("Removal probability by position", fontsize=11)
axes_inf[1].set_xticks([0,1,2]); axes_inf[1].set_xticklabels(["Left","Centre","Right"])
plt.colorbar(im, ax=axes_inf[1], shrink=0.8, label="P(removed)")
# Mark top predictions
for bid in removed_ids[:n_removed_pred]:
    try:
        ly = int(bid[1:3]); sl = int(bid.split("_")[-1])
        axes_inf[1].plot(sl, ly, "w*", markersize=12, markeredgecolor="k", markeredgewidth=0.8)
    except: pass

# Right panel: prediction summary
ax_txt = axes_inf[2]; ax_txt.set_axis_off()
summary_lines = [
    ("Modelo",            used_model.split("(")[0].strip()),
    ("",                  ""),
    ("Blocks removed",    f"{n_removed_pred}  (raw: {pred_num_removed:.2f})"),
    ("COM imbalance",     f"{pred_imbalance*1000:.2f} mm"),
    ("Torque risk",       f"{torque_pct:.1f}%  →  {risk_label.upper()}"),
    ("",                  ""),
    ("Ziglar Fapp_min",   f"{fapp_min_pred:.1f} mN  (Eq. 4)"),
    ("Ziglar Fapp_tor",   f"{fapp_tor_pred:.1f} mN  (Eq. 8)"),
    ("µs configured",     f"{FRICTION_LEVELS[TEST_MU_LEVEL]:.2f}  ({TEST_MU_LEVEL})"),
    ("",                  ""),
    ("Predicted blocks",  ""),
]
y_pos = 0.97; ax_txt.set_xlim(0, 1); ax_txt.set_ylim(0, 1)
risk_color = _RISK_COLORS[risk_label]
for label, value in summary_lines:
    if label:
        vc = risk_color if "torque" in label.lower() else "#1a1a2e"
        ax_txt.text(0.02, y_pos, f"{label}:", fontsize=9, color="#555555",
                    va="top", transform=ax_txt.transAxes)
        ax_txt.text(0.48, y_pos, value, fontsize=9, color=vc,
                    va="top", fontweight="bold" if "torque" in label.lower() else "normal",
                    transform=ax_txt.transAxes)
    y_pos -= 0.078

# Listar los bloques predichos (abreviado)
for k, bid in enumerate(removed_ids[:6]):
    try:
        ly = int(bid[1:3]); sl = int(bid.split("_")[-1])
        slot_name = ["Left","Centre","Right"][sl]
        p_val = pred_locs_prob[ly*3+sl]
        ax_txt.text(0.05, y_pos, f"  {bid}  layer {ly} {slot_name}  p={p_val:.2f}",
                    fontsize=8, color="#2c3e50", va="top", transform=ax_txt.transAxes)
        y_pos -= 0.065
    except: pass
if len(removed_ids) > 6:
    ax_txt.text(0.05, y_pos, f"  … and {len(removed_ids)-6} more",
                fontsize=8, color="#888", va="top", transform=ax_txt.transAxes)

ax_txt.set_title("Model predictions", fontsize=11)
fig_inf.patch.set_facecolor("#F8F4EE")
fig_inf.tight_layout()
panel_path = f"data/dashboard/inference_{_stem}.png"
fig_inf.savefig(panel_path, dpi=120, bbox_inches="tight", facecolor="#F8F4EE")
plt.close(fig_inf)


# ── 7. Display results ───────────────────────────────────────────────────────
from IPython.display import Image as _Img, Video as _Vid, display

print(f"\n{'='*60}")
print(f"  Inference panel     : {panel_path}")
print(f"  Reconstruction video: {out_video_path}")
sz_mb = os.path.getsize(out_video_path) / 1024**2 if os.path.exists(out_video_path) else 0
print(f"  Video size          : {sz_mb:.1f} MB")
print(f"{'='*60}")

display(_Img(panel_path))

if os.path.exists(out_video_path) and sz_mb > 0.01:
    print("\n  Reconstruction video (forward → pause → reverse):")
    display(_Vid(out_video_path, embed=True, width=480))
else:
    print("  ⚠ Video not generated — check errors above")
